# **Install Packages**

> faiss-cpu: Enables embeddings with optimized CPU implementations.


>transformers: Provides thousands of pretrained NLP models (like BERT/GPT)

> sentence_transformers: Generates semantic-aware sentence embeddings optimized for similarity comparison and retrieval tasks.

> pandas: Offers powerful data structures and tools for efficient data manipulation

In [20]:
!pip install faiss-cpu
!pip install transformers
!pip install sentence_transformers
!pip install pandas

# **Import Libraries**

In [21]:
import json
import re
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline

# **1. Load multilingual embedding model + FAISS index + context data**


```
This cell does:

1. Data Loading: Reads a JSON file.
2. English Extraction: Isolates only the English text contexts for processing
3. Embedding Generation: Uses a multilingual sentence transformer model to convert text to numerical vectors.
4. Index Creation: Builds a FAISS search index for fast similarity searches.
5. Index Saving: Stores the index to disk for later reuse

Key Purpose: This prepares a searchable database of English text embeddings that can quickly find relevant contexts when given a query.

Technical Note: Uses `paraphrase-multilingual-MiniLM-L12-v2` model which handles multiple languages but only English texts are being indexed here. The L2 (Euclidean) distance metric is used for similarity comparisons.
```



In [22]:
with open('/content/data.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# 2. Extract just the English contexts (for embedding/indexing)
contexts_en = [entry['context']['en'] for entry in raw_data]

# 3. Encode them
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = model.encode(contexts_en, convert_to_numpy=True)

# 4. Build + save a new FAISS index
d = embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(embeddings)
faiss.write_index(index, 'faiss_index.bin')

print(f"Index rebuilt with {len(contexts_en)} contexts.")

Index rebuilt with 266 contexts.


# **2. Utility to detect Arabic vs. English based on Unicode ranges**

```
What it does:
Detects if text contains Arabic script characters, returning 'ar' if found or 'en' otherwise.

Key Characteristics:

1. Uses regex to check for Arabic Unicode characters (\u0600-\u06FF)

2. Binary output: 'ar' (if Arabic detected) or 'en' (default)

3. Fast, lightweight, no dependencies

4. Assumes all non-Arabic is English
```

In [23]:
def detect_language(text: str) -> str:
    if re.search(r'[\u0600-\u06FF]', text):
        return 'ar'
    return 'en'

# **3. Retrieve top‑k contexts (still returns full entry + distance)**


```
What it does:
1. Finds the top 'k' most semantically similar contexts to a query using vector search.

2. Encodes the query into a numerical embedding.

3. Searches a pre-built FAISS index for the closest matches.

4. Returns the original bilingual entries (English + Arabic) with their similarity scores.
```

In [24]:
def search_context(query: str, k: int = 3):
    q_emb = model.encode([query])
    distances, indices = index.search(np.array(q_emb), k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        entry = raw_data[idx]        # the full bilingual dict
        results.append((entry, dist))
    return results

# **4. Generate answer with HuggingFace QA pipeline, selecting sub‑fields**


```
What it does:
Purpose: Retrieves a pre-stored answer (in English or Arabic) that best matches the user's question, using semantic search.

Finds the best pre-written answer for a question by:
1. Detecting the question's language (en/ar)

2. Retrieving the most similar context via semantic search

3. Returning the matching answer in the same language
```

In [25]:
qa = pipeline("question-answering", model="bert-base-multilingual-cased")

def generate_answer(query: str):
    # 1) detect language
    lang = detect_language(query)

    # 2) retrieve the top match
    entry, dist = search_context(query, k=1)[0]

    # 3) pull the stored answer (not the QA-model output)
    answer_text = entry['answer'][lang]
    ctx_text    = entry['context'][lang]

    return {
        'lang':    lang,
        'query':   query,
        'context': ctx_text,
        'answer':  answer_text,
        'dist':    dist
    }

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu


# **5. Main loop**

```
What it does:
Features:

1.Auto-detects language (Arabic/English) from input
2. Two modes:

      1: Show top 3 related contexts + similarity scores

      2: Retrieve pre-stored answer + supporting context
3.Type exit to quit anytime
---------------------------------------------------------
How it works:

1. Asks for mode (1 or 2)
2. Takes user's question
3. Detects language (Arabic if contains Arabic script, else English)
4. Returns results in the detected language
```

In [26]:
def contains_arabic(text):
    # Check if text contains any Arabic characters (Unicode range)
    for char in text:
        if '\u0600' <= char <= '\u06FF':
            return True
    return False

def main():
    print("Welcome! Type 'exit' at any prompt to quit.")
    print("مرحبًا! اكتب 'exit' في أي وقت للخروج.")

    while True:
        print("-------------------------------------------")
        # 1) Ask for mode with bilingual prompt
        mode_input = input(
            "اختر الوضع (1 = عرض السياقات، 2 = إجابة سؤال)"
            "\nSelect mode (1: View contexts, 2: Answer a question): "
        ).strip()

        if mode_input.lower() == 'exit':
            break

        print("-------------------------------------------")
        # 2) Ask for the actual question with bilingual prompt
        query = input(
            "\nAsk your question / ادخل سؤالك: "
        ).strip()

        if query.lower() == 'exit':
            break

        # Detect language from question content
        lang = 'ar' if contains_arabic(query) else 'en'

        # 3) Branch on mode
        if mode_input == '1':
            hits = search_context(query, k=3)
            header = "نتائج البحث:" if lang == 'ar' else "Search results:"
            print(header)
            for i, (entry, dist) in enumerate(hits, 1):
                label = "النص" if lang == 'ar' else "Context"
                print(f"{label} {i}: {entry['context'][lang]} (distance: {dist:.4f})")

        elif mode_input == '2':
            res = generate_answer(query)
            q_lbl = "السؤال"        if lang == 'ar' else "Question"
            c_lbl = "النص المستخدم" if lang == 'ar' else "Context used"
            a_lbl = "الإجابة"       if lang == 'ar' else "Answer"

            print(f"\n{q_lbl}: {res['query']}")
            print(f"{c_lbl}: {res['context']}")
            print(f"{a_lbl}: {res['answer']}")

        else:
            # Bilingual error message for invalid mode
            print("Invalid choice! / اختيار غير صحيح!")

if __name__ == "__main__":
    main()

Welcome! Type 'exit' at any prompt to quit.
مرحبًا! اكتب 'exit' في أي وقت للخروج.
-------------------------------------------
اختر الوضع (1 = عرض السياقات، 2 = إجابة سؤال)
Select mode (1: View contexts, 2: Answer a question): 1
-------------------------------------------

Ask your question / ادخل سؤالك: ما هي سرعة الضوء؟
نتائج البحث:
النص 1: تبلغ سرعة الضوء حوالي 299,792 كيلومتر في الثانية، مما يجعلها أسرع شيء في الكون. (distance: 10.2208)
النص 2: تبلغ سرعة الضوء حوالي 299,792 كيلومتر في الثانية، مما يجعلها أسرع شيء في الكون. (distance: 10.2208)
النص 3: الفوتونيات هي علم توليد واكتشاف ومعالجة الضوء، تُستخدم في الاتصالات والتصوير. (distance: 23.8074)
-------------------------------------------
اختر الوضع (1 = عرض السياقات، 2 = إجابة سؤال)
Select mode (1: View contexts, 2: Answer a question): 2
-------------------------------------------

Ask your question / ادخل سؤالك: What do antipyretics do?

Question: What do antipyretics do?
Context used: Antipyretics are medications that reduce fev